# JSON Kafka events

Kafka keys and values are sent as encoded bytes, and this notebook writes three event records to `events.json`.

Those records land in a few tables: a raw table, a decoded table, and then one table for each way of reading the JSON.

The three main JSON handling methods are a regular flatten, a struct built from the JSON schema with `from_json`, and a variant produced by `parse_json`.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA data_ingestion;

CREATE VOLUME IF NOT EXISTS raw_files;

In [0]:
import json
import base64

events = [
    {
        "key": "user-101",
        "value": {
            "device": "iOS",
            "event_name": "add_item",
            "geo": {
                "city": "Helsinki",
                "state": "Uusimaa"
            },
            "items": [
                {"item_id": "SKU-101", "quantity": 2}
            ]
        },
        "offset": 100,
        "partition": 0,
        "timestamp": 1760000000000,
        "topic": "ecommerce-events"
    },
    {
        "key": "user-102",
        "value": {
            "device": "Android",
            "event_name": "purchase",
            "geo": {
                "city": "Espoo",
                "state": "Uusimaa"
            },
            "items": [
                {"item_id": "SKU-202", "quantity": 1}
            ]
        },
        "offset": 101,
        "partition": 0,
        "timestamp": 1760000001000,
        "topic": "ecommerce-events"
    },
    {
        "key": "user-103",
        "value": {
            "device": "Web",
            "event_name": "view_item",
            "geo": {
                "city": "Tampere",
                "state": "Pirkanmaa"
            },
            "items": [
                {"item_id": "SKU-303", "quantity": 3}
            ]
        },
        "offset": 102,
        "partition": 0,
        "timestamp": 1760000002000,
        "topic": "ecommerce-events"
    }
]

def encode_base64(value):
    if not isinstance(value, str):
        value = json.dumps(value)
    return base64.b64encode(value.encode("utf-8")).decode("utf-8")

records = []

for event in events:
    records.append({
        "key": encode_base64(event["key"]),
        "value": encode_base64(event["value"]),
        "offset": event["offset"],
        "partition": event["partition"],
        "timestamp": event["timestamp"],
        "topic": event["topic"]
    })

path = "/Volumes/workspace/data_ingestion/raw_files/events_kafka/events.json"

# JSON Lines format: one complete JSON object per line
json_lines = "\n".join(json.dumps(r) for r in records) + "\n"

dbutils.fs.put(path, json_lines, overwrite=True)

print("Created 3 Kafka-style JSON records")

# Operation	            Input	        Output
# json.dumps()	        Python object	JSON string
# json.loads()	        JSON string	    Python object
# .encode("utf-8")	    String	        Bytes
# .decode("utf-8")	    Bytes	        String
# base64.b64encode()	Bytes	        Base64 bytes
# base64.b64decode()	Base64 bytes	Original bytes

Wrote 951 bytes.
Created 3 Kafka-style JSON records


In [0]:
%sql
LIST '/Volumes/workspace/data_ingestion/raw_files/events_kafka/';

path,name,size,modification_time
/Volumes/workspace/data_ingestion/raw_files/events_kafka/events.json,events.json,951,1791716689000


In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/data_ingestion/raw_files/events_kafka/',
  format => 'json'
)
LIMIT 10;

key,offset,partition,timestamp,topic,value,_rescued_data
dXNlci0xMDE=,100,0,1760000000000,ecommerce-events,eyJkZXZpY2UiOiAiaU9TIiwgImV2ZW50X25hbWUiOiAiYWRkX2l0ZW0iLCAiZ2VvIjogeyJjaXR5IjogIkhlbHNpbmtpIiwgInN0YXRlIjogIlV1c2ltYWEifSwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJTS1UtMTAxIiwgInF1YW50aXR5IjogMn1dfQ==,null
dXNlci0xMDI=,101,0,1760000001000,ecommerce-events,eyJkZXZpY2UiOiAiQW5kcm9pZCIsICJldmVudF9uYW1lIjogInB1cmNoYXNlIiwgImdlbyI6IHsiY2l0eSI6ICJFc3BvbyIsICJzdGF0ZSI6ICJVdXNpbWFhIn0sICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiU0tVLTIwMiIsICJxdWFudGl0eSI6IDF9XX0=,null
dXNlci0xMDM=,102,0,1760000002000,ecommerce-events,eyJkZXZpY2UiOiAiV2ViIiwgImV2ZW50X25hbWUiOiAidmlld19pdGVtIiwgImdlbyI6IHsiY2l0eSI6ICJUYW1wZXJlIiwgInN0YXRlIjogIlBpcmthbm1hYSJ9LCAiaXRlbXMiOiBbeyJpdGVtX2lkIjogIlNLVS0zMDMiLCAicXVhbnRpdHkiOiAzfV19,null


In [0]:
%sql
CREATE OR REPLACE TABLE kafka_events_bronze_raw AS
SELECT *
FROM read_files(
  '/Volumes/workspace/data_ingestion/raw_files/events_kafka/',
  format => 'json'
);

SELECT * FROM kafka_events_bronze_raw;

key,offset,partition,timestamp,topic,value,_rescued_data
dXNlci0xMDE=,100,0,1760000000000,ecommerce-events,eyJkZXZpY2UiOiAiaU9TIiwgImV2ZW50X25hbWUiOiAiYWRkX2l0ZW0iLCAiZ2VvIjogeyJjaXR5IjogIkhlbHNpbmtpIiwgInN0YXRlIjogIlV1c2ltYWEifSwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJTS1UtMTAxIiwgInF1YW50aXR5IjogMn1dfQ==,null
dXNlci0xMDI=,101,0,1760000001000,ecommerce-events,eyJkZXZpY2UiOiAiQW5kcm9pZCIsICJldmVudF9uYW1lIjogInB1cmNoYXNlIiwgImdlbyI6IHsiY2l0eSI6ICJFc3BvbyIsICJzdGF0ZSI6ICJVdXNpbWFhIn0sICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiU0tVLTIwMiIsICJxdWFudGl0eSI6IDF9XX0=,null
dXNlci0xMDM=,102,0,1760000002000,ecommerce-events,eyJkZXZpY2UiOiAiV2ViIiwgImV2ZW50X25hbWUiOiAidmlld19pdGVtIiwgImdlbyI6IHsiY2l0eSI6ICJUYW1wZXJlIiwgInN0YXRlIjogIlBpcmthbm1hYSJ9LCAiaXRlbXMiOiBbeyJpdGVtX2lkIjogIlNLVS0zMDMiLCAicXVhbnRpdHkiOiAzfV19,null


In [0]:
%sql
SELECT
    key,
    unbase64(key) AS decoded_key_binary,
    CAST(unbase64(key) AS STRING) AS decoded_key,

    value,
    CAST(unbase64(value) AS STRING) AS decoded_value

FROM kafka_events_bronze_raw
LIMIT 5;

key,decoded_key_binary,decoded_key,value,decoded_value
dXNlci0xMDE=,dXNlci0xMDE=,user-101,eyJkZXZpY2UiOiAiaU9TIiwgImV2ZW50X25hbWUiOiAiYWRkX2l0ZW0iLCAiZ2VvIjogeyJjaXR5IjogIkhlbHNpbmtpIiwgInN0YXRlIjogIlV1c2ltYWEifSwgIml0ZW1zIjogW3siaXRlbV9pZCI6ICJTS1UtMTAxIiwgInF1YW50aXR5IjogMn1dfQ==,"{""device"": ""iOS"", ""event_name"": ""add_item"", ""geo"": {""city"": ""Helsinki"", ""state"": ""Uusimaa""}, ""items"": [{""item_id"": ""SKU-101"", ""quantity"": 2}]}"
dXNlci0xMDI=,dXNlci0xMDI=,user-102,eyJkZXZpY2UiOiAiQW5kcm9pZCIsICJldmVudF9uYW1lIjogInB1cmNoYXNlIiwgImdlbyI6IHsiY2l0eSI6ICJFc3BvbyIsICJzdGF0ZSI6ICJVdXNpbWFhIn0sICJpdGVtcyI6IFt7Iml0ZW1faWQiOiAiU0tVLTIwMiIsICJxdWFudGl0eSI6IDF9XX0=,"{""device"": ""Android"", ""event_name"": ""purchase"", ""geo"": {""city"": ""Espoo"", ""state"": ""Uusimaa""}, ""items"": [{""item_id"": ""SKU-202"", ""quantity"": 1}]}"
dXNlci0xMDM=,dXNlci0xMDM=,user-103,eyJkZXZpY2UiOiAiV2ViIiwgImV2ZW50X25hbWUiOiAidmlld19pdGVtIiwgImdlbyI6IHsiY2l0eSI6ICJUYW1wZXJlIiwgInN0YXRlIjogIlBpcmthbm1hYSJ9LCAiaXRlbXMiOiBbeyJpdGVtX2lkIjogIlNLVS0zMDMiLCAicXVhbnRpdHkiOiAzfV19,"{""device"": ""Web"", ""event_name"": ""view_item"", ""geo"": {""city"": ""Tampere"", ""state"": ""Pirkanmaa""}, ""items"": [{""item_id"": ""SKU-303"", ""quantity"": 3}]}"


In [0]:
%sql
CREATE OR REPLACE TABLE kafka_events_bronze_decoded AS
SELECT
    CAST(unbase64(key) AS STRING) AS decoded_key,
    offset,
    partition,
    timestamp,
    topic,
    CAST(unbase64(value) AS STRING) AS decoded_value
FROM kafka_events_bronze_raw;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    decoded_key,
    decoded_value
FROM kafka_events_bronze_decoded;

decoded_key,decoded_value
user-101,"{""device"": ""iOS"", ""event_name"": ""add_item"", ""geo"": {""city"": ""Helsinki"", ""state"": ""Uusimaa""}, ""items"": [{""item_id"": ""SKU-101"", ""quantity"": 2}]}"
user-102,"{""device"": ""Android"", ""event_name"": ""purchase"", ""geo"": {""city"": ""Espoo"", ""state"": ""Uusimaa""}, ""items"": [{""item_id"": ""SKU-202"", ""quantity"": 1}]}"
user-103,"{""device"": ""Web"", ""event_name"": ""view_item"", ""geo"": {""city"": ""Tampere"", ""state"": ""Pirkanmaa""}, ""items"": [{""item_id"": ""SKU-303"", ""quantity"": 3}]}"


In [0]:
%sql
SELECT
    decoded_key,
    decoded_value:device AS device,
    decoded_value:event_name AS event_name,
    decoded_value:geo.city AS city,
    decoded_value:geo.state AS state,
    decoded_value:items AS items
FROM kafka_events_bronze_decoded;

decoded_key,device,event_name,city,state,items
user-101,iOS,add_item,Helsinki,Uusimaa,"[{""item_id"":""SKU-101"",""quantity"":2}]"
user-102,Android,purchase,Espoo,Uusimaa,"[{""item_id"":""SKU-202"",""quantity"":1}]"
user-103,Web,view_item,Tampere,Pirkanmaa,"[{""item_id"":""SKU-303"",""quantity"":3}]"


In [0]:
%sql
CREATE OR REPLACE TABLE kafka_events_bronze_flattened AS
SELECT
    decoded_key AS user_id,
    offset,
    partition,
    timestamp,
    topic,
    decoded_value:device AS device,
    decoded_value:event_name AS event_name,
    decoded_value:geo.city AS city,
    decoded_value:geo.state AS state,
    decoded_value:items AS items
FROM kafka_events_bronze_decoded;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM kafka_events_bronze_flattened;

user_id,offset,partition,timestamp,topic,device,event_name,city,state,items
user-101,100,0,1760000000000,ecommerce-events,iOS,add_item,Helsinki,Uusimaa,"[{""item_id"":""SKU-101"",""quantity"":2}]"
user-102,101,0,1760000001000,ecommerce-events,Android,purchase,Espoo,Uusimaa,"[{""item_id"":""SKU-202"",""quantity"":1}]"
user-103,102,0,1760000002000,ecommerce-events,Web,view_item,Tampere,Pirkanmaa,"[{""item_id"":""SKU-303"",""quantity"":3}]"


In [0]:
%sql
SELECT
    decoded_key AS user_id,
    decoded_value:items[0].item_id AS item_id,
    CAST(decoded_value:items[0].quantity AS INT) AS quantity
FROM kafka_events_bronze_decoded;

user_id,item_id,quantity
user-101,SKU-101,2
user-102,SKU-202,1
user-103,SKU-303,3


JSON string to STRUCT

In [0]:
%sql
SELECT schema_of_json(
  '{
    "device": "iOS",
    "event_name": "add_item",
    "geo": {"city": "Helsinki", "state": "Uusimaa"},
    "items": [{"item_id": "SKU-101", "quantity": 2}]
  }'
) AS inferred_schema;

inferred_schema
"STRUCT, items: ARRAY>>"


In [0]:
%sql
CREATE OR REPLACE TABLE kafka_events_bronze_struct AS
SELECT
    decoded_key,
    offset,
    partition,
    timestamp,
    topic,

    from_json(
        decoded_value,
        'STRUCT<
            device: STRING,
            event_name: STRING,
            geo: STRUCT<city: STRING, state: STRING>,
            items: ARRAY<STRUCT<item_id: STRING, quantity: BIGINT>>
        >'
    ) AS value

FROM kafka_events_bronze_decoded;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM kafka_events_bronze_struct;

decoded_key,offset,partition,timestamp,topic,value
user-101,100,0,1760000000000,ecommerce-events,"List(iOS, add_item, List(Helsinki, Uusimaa), List(List(SKU-101, 2)))"
user-102,101,0,1760000001000,ecommerce-events,"List(Android, purchase, List(Espoo, Uusimaa), List(List(SKU-202, 1)))"
user-103,102,0,1760000002000,ecommerce-events,"List(Web, view_item, List(Tampere, Pirkanmaa), List(List(SKU-303, 3)))"


In [0]:
%sql
DESCRIBE TABLE kafka_events_bronze_struct;

col_name,data_type,comment
decoded_key,string,null
offset,bigint,null
partition,bigint,null
timestamp,bigint,null
topic,string,null
value,"struct,items:array>>",null


In [0]:
%sql
SELECT
    decoded_key,
    value.device AS device,
    value.event_name AS event_name,
    value.geo.city AS city,
    value.geo.state AS state,
    value.items AS items,
    array_size(value.items) AS item_count
FROM kafka_events_bronze_struct;

decoded_key,device,event_name,city,state,items,item_count
user-101,iOS,add_item,Helsinki,Uusimaa,"List(List(SKU-101, 2))",1
user-102,Android,purchase,Espoo,Uusimaa,"List(List(SKU-202, 1))",1
user-103,Web,view_item,Tampere,Pirkanmaa,"List(List(SKU-303, 3))",1


JSON stored as text: 
decoded_value:geo.city

JSON parsed into STRUCT:
value.geo.city

In [0]:
%sql
INSERT INTO kafka_events_bronze_struct
SELECT
    'user-104',
    103L,
    0,
    1760000003000L,
    'ecommerce-events',
    named_struct(
        'device', 'iOS',
        'event_name', 'purchase',
        'geo', named_struct(
            'city', 'Turku',
            'state', 'Varsinais-Suomi'
        ),
        'items', array(
            named_struct('item_id', 'SKU-401', 'quantity', 2L),
            named_struct('item_id', 'SKU-402', 'quantity', 1L),
            named_struct('item_id', 'SKU-403', 'quantity', 4L)
        )
    );

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT
    decoded_key,
    array_size(value.items) AS item_count
FROM kafka_events_bronze_struct
ORDER BY item_count DESC;

decoded_key,item_count
user-104,3
user-102,1
user-101,1
user-103,1


In [0]:
%sql
CREATE OR REPLACE TABLE bronze_exploded_array AS
SELECT
    decoded_key,
    value.device AS device,
    value.geo.city AS city,
    array_size(value.items) AS number_of_items,
    item.col.item_id AS item_id,
    item.col.quantity AS quantity
FROM kafka_events_bronze_struct,
LATERAL explode(value.items) AS item;

-- %sql
-- CREATE OR REPLACE TABLE bronze_exploded_array AS
-- SELECT
--     decoded_key,
--     value.device AS device,
--     value.geo.city AS city,
--     array_size(value.items) AS number_of_items,
--     exploded_item.item_id AS item_id,
--     exploded_item.quantity AS quantity
-- FROM kafka_events_bronze_struct,
-- LATERAL explode(value.items) AS e(exploded_item);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM bronze_exploded_array
ORDER BY decoded_key, item_id;

decoded_key,device,city,number_of_items,item_id,quantity
user-101,iOS,Helsinki,1,SKU-101,2
user-102,Android,Espoo,1,SKU-202,1
user-103,Web,Tampere,1,SKU-303,3
user-104,iOS,Turku,3,SKU-401,2
user-104,iOS,Turku,3,SKU-402,1
user-104,iOS,Turku,3,SKU-403,4


explode() removes rows whose arrays are empty or NULL.

explode_outer() preserves a row for empty or NULL arrays, with NULL exploded values.

## VARIANT
VARIANT is a Databricks data type that stores semi-structured data, such as JSON, without requiring you to define every field and its type in advance.

In [0]:
%sql
SELECT
    decoded_key,
    decoded_value
FROM kafka_events_bronze_decoded
LIMIT 5;

decoded_key,decoded_value
user-101,"{""device"": ""iOS"", ""event_name"": ""add_item"", ""geo"": {""city"": ""Helsinki"", ""state"": ""Uusimaa""}, ""items"": [{""item_id"": ""SKU-101"", ""quantity"": 2}]}"
user-102,"{""device"": ""Android"", ""event_name"": ""purchase"", ""geo"": {""city"": ""Espoo"", ""state"": ""Uusimaa""}, ""items"": [{""item_id"": ""SKU-202"", ""quantity"": 1}]}"
user-103,"{""device"": ""Web"", ""event_name"": ""view_item"", ""geo"": {""city"": ""Tampere"", ""state"": ""Pirkanmaa""}, ""items"": [{""item_id"": ""SKU-303"", ""quantity"": 3}]}"


In [0]:
%sql
CREATE OR REPLACE TABLE kafka_events_bronze_variant AS
SELECT
    decoded_key,
    offset,
    partition,
    timestamp,
    topic,
    parse_json(decoded_value) AS json_variant_value
FROM kafka_events_bronze_decoded;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM kafka_events_bronze_variant
LIMIT 5;

decoded_key,offset,partition,timestamp,topic,json_variant_value
user-101,100,0,1760000000000,ecommerce-events,"{""device"":""iOS"",""event_name"":""add_item"",""geo"":{""city"":""Helsinki"",""state"":""Uusimaa""},""items"":[{""item_id"":""SKU-101"",""quantity"":2}]}"
user-102,101,0,1760000001000,ecommerce-events,"{""device"":""Android"",""event_name"":""purchase"",""geo"":{""city"":""Espoo"",""state"":""Uusimaa""},""items"":[{""item_id"":""SKU-202"",""quantity"":1}]}"
user-103,102,0,1760000002000,ecommerce-events,"{""device"":""Web"",""event_name"":""view_item"",""geo"":{""city"":""Tampere"",""state"":""Pirkanmaa""},""items"":[{""item_id"":""SKU-303"",""quantity"":3}]}"


In [0]:
%sql
SELECT
    decoded_key,
    json_variant_value:device::STRING AS device,
    json_variant_value:event_name::STRING AS event_name,
    json_variant_value:geo.city::STRING AS city,
    json_variant_value:items AS items
FROM kafka_events_bronze_variant;

decoded_key,device,event_name,city,items
user-101,iOS,add_item,Helsinki,"[{""item_id"":""SKU-101"",""quantity"":2}]"
user-102,Android,purchase,Espoo,"[{""item_id"":""SKU-202"",""quantity"":1}]"
user-103,Web,view_item,Tampere,"[{""item_id"":""SKU-303"",""quantity"":3}]"


In [0]:
%sql
SELECT
    decoded_key,
    json_variant_value:items[0].item_id::STRING AS item_id,
    json_variant_value:items[0].quantity::INT AS quantity
FROM kafka_events_bronze_variant;

decoded_key,item_id,quantity
user-101,SKU-101,2
user-102,SKU-202,1
user-103,SKU-303,3
